# Brazilian Wage Analysis 2012-2025: Interactive Notebook

**Author:** Vitor Ramos dos Santos  
**Date:** February 2026  
**Version:** 1.0

---

## Overview

This notebook provides an interactive exploration of Brazilian wage dynamics from 2012 to 2025, with advanced forecasting capabilities.

**Key Features:**
- Historical data visualization
- Statistical analysis (regression, correlations)
- Monte Carlo simulation (10,000 scenarios)
- Interactive scenario builder
- Stress testing

**Data Sources:**
- IBGE PNAD Contínua (household survey)
- Brazilian Ministry of Labor (CAGED employment data)
- National Accounts (GDP, wage mass)

---

## Setup: Install Dependencies and Load Data

In [ ]:
# Install required packages
!pip install -q pandas numpy matplotlib seaborn plotly ipywidgets

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from ipywidgets import interact, FloatSlider, IntSlider
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

# Configuration
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete. Ready to analyze.')

In [ ]:
# Load data from GitHub repository
BASE_URL = 'https://raw.githubusercontent.com/Vitor2316/Projeto-analise-de-dados-Brasil/main/dados/'

# Historical data
df_brasil = pd.read_csv(BASE_URL + 'brasil_anual_CORRIGIDO_FINAL.csv')
df_percentis = pd.read_csv(BASE_URL + 'percentis_rendimento.csv')
df_desemprego = pd.read_csv(BASE_URL + 'desemprego_salario.csv')
df_pib = pd.read_csv(BASE_URL + 'participacao_pib.csv')

print('Data loaded successfully.')
print(f'Historical period: {df_brasil["ano"].min()}-{df_brasil["ano"].max()}')
print(f'Records: {len(df_brasil)}')

---

## Section 1: Historical Data Exploration

In [ ]:
# Display basic statistics
print('Median Wage Statistics (2012-2024):')
print('='*50)
print(f'Initial (2012): R${df_percentis["p50"].iloc[0]}')
print(f'Final (2024): R${df_percentis["p50"].iloc[-1]}')
print(f'Total change: {((df_percentis["p50"].iloc[-1]/df_percentis["p50"].iloc[0])-1)*100:.1f}%')
print(f'Min: R${df_percentis["p50"].min()} in {df_percentis.loc[df_percentis["p50"].idxmin(), "ano"]}')
print(f'Max: R${df_percentis["p50"].max()} in {df_percentis.loc[df_percentis["p50"].idxmax(), "ano"]}')

In [ ]:
# Interactive visualization: Wage trajectory
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_percentis['ano'],
    y=df_percentis['p50'],
    mode='lines+markers',
    name='Median Wage (P50)',
    line=dict(color='#2E86C1', width=3),
    marker=dict(size=8)
))

# Reference line
fig.add_hline(
    y=df_percentis['p50'].iloc[0],
    line_dash='dash',
    line_color='gray',
    annotation_text='2012 baseline',
    annotation_position='right'
)

# Mark critical points
fig.add_annotation(
    x=2014, y=865,
    text='Peak',
    showarrow=True,
    arrowhead=2,
    ax=0, ay=-40
)

fig.add_annotation(
    x=2021, y=810,
    text='Crisis low<br>(back to 2012 level)',
    showarrow=True,
    arrowhead=2,
    ax=40, ay=20
)

fig.update_layout(
    title='Real Wage Trajectory: Brazilian Workers (2012-2024)',
    xaxis_title='Year',
    yaxis_title='Real Median Wage (R$ 2012)',
    hovermode='x unified',
    height=500
)

fig.show()

In [ ]:
# Distributional analysis
fig = go.Figure()

fig.add_trace(go.Scatter(x=df_percentis['ano'], y=df_percentis['p10'], 
                         name='P10 (bottom 10%)', line=dict(color='#E74C3C')))
fig.add_trace(go.Scatter(x=df_percentis['ano'], y=df_percentis['p50'], 
                         name='P50 (median)', line=dict(color='#3498DB')))
fig.add_trace(go.Scatter(x=df_percentis['ano'], y=df_percentis['p90'], 
                         name='P90 (top 10%)', line=dict(color='#2ECC71')))

fig.update_layout(
    title='Wage Distribution: Progressive Gains',
    xaxis_title='Year',
    yaxis_title='Real Wage (R$ 2012)',
    hovermode='x unified',
    height=500
)

fig.show()

# Calculate growth by percentile
print('\nGrowth by Income Percentile (2012-2024):')
print('='*50)
for col in ['p10', 'p50', 'p90']:
    growth = ((df_percentis[col].iloc[-1]/df_percentis[col].iloc[0])-1)*100
    print(f'{col.upper()}: {growth:+.1f}%')

print('\nConclusion: Base grew more than top (progressive distribution)')

---

## Section 2: Statistical Analysis

In [ ]:
# Regression: Unemployment vs Wage
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

X = df_desemprego['desemprego'].values.reshape(-1, 1)
y = df_desemprego['p50_real'].values

model = LinearRegression()
model.fit(X, y)

y_pred = model.predict(X)
r2 = r2_score(y, y_pred)

print('Regression Analysis: Unemployment → Wage')
print('='*50)
print(f'Coefficient: {model.coef_[0]:.2f} R$ per pp unemployment')
print(f'Intercept: {model.intercept_:.2f}')
print(f'R²: {r2:.3f}')
print(f'\nInterpretation: Each 1pp increase in unemployment')
print(f'reduces median wage by R${abs(model.coef_[0]):.2f}')

# Calculate p-value
n = len(y)
residuals = y - y_pred
std_residuals = np.std(residuals)
x_mean = np.mean(X)
x_var = np.sum((X - x_mean)**2)
se_coef = std_residuals / np.sqrt(x_var)
t_stat = model.coef_[0] / se_coef
p_value = 2 * (1 - stats.t.cdf(abs(t_stat), n-2))

print(f'\nStatistical significance: p-value = {p_value:.4f}')
if p_value < 0.05:
    print('Result is STATISTICALLY SIGNIFICANT (p < 0.05)')
else:
    print('Result is NOT statistically significant (p >= 0.05)')

In [ ]:
# Visualization: Unemployment vs Wage with regression line
fig = go.Figure()

# Scatter plot
fig.add_trace(go.Scatter(
    x=df_desemprego['desemprego'],
    y=df_desemprego['p50_real'],
    mode='markers',
    name='Observed',
    marker=dict(size=10, color='#3498DB'),
    text=df_desemprego['ano'],
    hovertemplate='Year: %{text}<br>Unemployment: %{x:.1f}%<br>Wage: R$%{y:.0f}<extra></extra>'
))

# Regression line
x_line = np.linspace(df_desemprego['desemprego'].min(), df_desemprego['desemprego'].max(), 100)
y_line = model.predict(x_line.reshape(-1, 1))

fig.add_trace(go.Scatter(
    x=x_line,
    y=y_line,
    mode='lines',
    name=f'Regression (R²={r2:.3f})',
    line=dict(color='#E74C3C', dash='dash')
))

fig.update_layout(
    title='Unemployment vs Median Wage: Inverse Relationship',
    xaxis_title='Unemployment Rate (%)',
    yaxis_title='Real Median Wage (R$ 2012)',
    hovermode='closest',
    height=500
)

fig.show()

In [ ]:
# Correlation matrix
df_corr = pd.DataFrame({
    'Wage': df_desemprego['p50_real'],
    'Unemployment': df_desemprego['desemprego'],
    'Labor_Share': df_pib['participacao_trabalho']
})

corr_matrix = df_corr.corr()

print('Correlation Matrix:')
print('='*50)
print(corr_matrix)
print('\nKey findings:')
print(f'- Wage vs Unemployment: {corr_matrix.loc["Wage", "Unemployment"]:.3f} (negative, as expected)')
print(f'- Wage vs Labor Share: {corr_matrix.loc["Wage", "Labor_Share"]:.3f} (positive)')

---

## Section 3: Interactive Scenario Builder

In [ ]:
# Model parameters
ELASTICIDADE_DESEMPREGO = -2.0
ELASTICIDADE_PIB = 0.3
ELASTICIDADE_SM = 0.4
ELASTICIDADE_INFLACAO = -0.5
BASE_2024 = 930

def calcular_salario(desemprego, pib, inflacao, sm_real):
    """Calculate projected wage based on economic parameters"""
    impacto_desemp = ELASTICIDADE_DESEMPREGO * (desemprego - 6.6)
    impacto_pib = ELASTICIDADE_PIB * pib
    impacto_sm = ELASTICIDADE_SM * sm_real
    impacto_inflacao = ELASTICIDADE_INFLACAO * (inflacao - 3.0)
    
    impacto_total = impacto_desemp + impacto_pib + impacto_sm + impacto_inflacao
    salario = BASE_2024 * (1 + impacto_total/100)
    
    return salario, impacto_total, {
        'Unemployment': impacto_desemp,
        'GDP': impacto_pib,
        'Min Wage': impacto_sm,
        'Inflation': impacto_inflacao
    }

# Interactive widget
@interact(
    desemprego=FloatSlider(value=7.0, min=5.0, max=15.0, step=0.5, description='Unemployment (%)'),
    pib=FloatSlider(value=2.0, min=-2.0, max=5.0, step=0.5, description='GDP Growth (%)'),
    inflacao=FloatSlider(value=5.5, min=3.0, max=10.0, step=0.5, description='Inflation (%)'),
    sm_real=FloatSlider(value=2.0, min=0.0, max=5.0, step=0.5, description='Min Wage Gain (%)')
)
def simular_cenario(desemprego, pib, inflacao, sm_real):
    salario, variacao, decomp = calcular_salario(desemprego, pib, inflacao, sm_real)
    
    print('\nProjected 2026 Wage:')
    print('='*50)
    print(f'R$ {salario:.2f} ({variacao:+.1f}% vs 2024)')
    print(f'\nImpact Decomposition:')
    for factor, impact in decomp.items():
        print(f'  {factor:12s}: {impact:+.2f}pp')
    
    # Visual feedback
    if salario >= BASE_2024 * 1.05:
        print('\nOutcome: STRONG GROWTH scenario')
    elif salario >= BASE_2024:
        print('\nOutcome: MODERATE GROWTH scenario')
    elif salario >= BASE_2024 * 0.95:
        print('\nOutcome: STABILITY scenario')
    else:
        print('\nOutcome: DECLINE scenario')
    
    # Compare to 2012
    vs_2012 = ((salario / 805) - 1) * 100
    print(f'\nVs 2012 baseline: {vs_2012:+.1f}%')

---

## Section 4: Monte Carlo Simulation

In [ ]:
# Run Monte Carlo simulation
def monte_carlo(n_sim=10000):
    """Run Monte Carlo simulation with n_sim scenarios"""
    np.random.seed(42)
    
    # Generate random parameters
    desemp = np.clip(np.random.normal(7.5, 1.5, n_sim), 5, 15)
    pib = np.clip(np.random.normal(2.0, 1.0, n_sim), -2, 5)
    inflacao = np.clip(np.random.normal(5.5, 1.0, n_sim), 3, 10)
    sm_real = np.clip(np.random.normal(2.0, 0.8, n_sim), 0, 5)
    
    # Calculate wages
    salarios = []
    for i in range(n_sim):
        sal, _, _ = calcular_salario(desemp[i], pib[i], inflacao[i], sm_real[i])
        salarios.append(sal)
    
    return np.array(salarios)

# Run simulation
print('Running Monte Carlo simulation (10,000 scenarios)...')
salarios_mc = monte_carlo(10000)
print('Complete.\n')

# Statistics
print('Results:')
print('='*50)
print(f'Mean: R$ {np.mean(salarios_mc):.2f}')
print(f'Median: R$ {np.median(salarios_mc):.2f}')
print(f'Std Dev: R$ {np.std(salarios_mc):.2f}')
print(f'5th percentile: R$ {np.percentile(salarios_mc, 5):.2f}')
print(f'95th percentile: R$ {np.percentile(salarios_mc, 95):.2f}')
print(f'\nProbability of decline vs 2024: {(salarios_mc < BASE_2024).sum()/len(salarios_mc)*100:.1f}%')
print(f'Probability of gain vs 2024: {(salarios_mc > BASE_2024).sum()/len(salarios_mc)*100:.1f}%')
print(f'Probability of decline >5%: {(salarios_mc < BASE_2024*0.95).sum()/len(salarios_mc)*100:.1f}%')

In [ ]:
# Visualize distribution
fig = go.Figure()

fig.add_trace(go.Histogram(
    x=salarios_mc,
    nbinsx=50,
    name='Distribution',
    marker_color='#3498DB',
    opacity=0.7
))

# Add vertical lines for key statistics
fig.add_vline(x=np.mean(salarios_mc), line_dash='dash', line_color='red',
              annotation_text=f'Mean: R${np.mean(salarios_mc):.0f}')
fig.add_vline(x=np.percentile(salarios_mc, 5), line_dash='dot', line_color='orange',
              annotation_text=f'P5: R${np.percentile(salarios_mc, 5):.0f}')
fig.add_vline(x=np.percentile(salarios_mc, 95), line_dash='dot', line_color='orange',
              annotation_text=f'P95: R${np.percentile(salarios_mc, 95):.0f}')
fig.add_vline(x=BASE_2024, line_dash='solid', line_color='green',
              annotation_text=f'2024 baseline: R${BASE_2024}')

fig.update_layout(
    title='Monte Carlo Simulation: Distribution of 2026 Wage Outcomes',
    xaxis_title='Projected 2026 Wage (R$ 2012)',
    yaxis_title='Frequency',
    height=500
)

fig.show()

---

## Section 5: Stress Testing

In [ ]:
# Define stress test scenarios
scenarios = {
    'Severe Crisis': {'desemprego': 12.0, 'pib': -2.0, 'inflacao': 8.0, 'sm_real': 0.0},
    'Stagflation': {'desemprego': 10.0, 'pib': 0.0, 'inflacao': 7.0, 'sm_real': 1.0},
    'Unsustainable Boom': {'desemprego': 5.0, 'pib': 4.0, 'inflacao': 6.0, 'sm_real': 3.0},
    'Recessionary Adjustment': {'desemprego': 9.0, 'pib': 0.5, 'inflacao': 5.0, 'sm_real': 1.5},
    'Base Case': {'desemprego': 7.0, 'pib': 2.0, 'inflacao': 5.5, 'sm_real': 2.0}
}

# Run all scenarios
results = []
for name, params in scenarios.items():
    sal, var, decomp = calcular_salario(**params)
    results.append({
        'Scenario': name,
        'Wage': sal,
        'Change (%)': var,
        'Unemployment': params['desemprego'],
        'GDP': params['pib'],
        'Inflation': params['inflacao']
    })

df_stress = pd.DataFrame(results)
df_stress = df_stress.sort_values('Wage', ascending=False)

print('Stress Test Results:')
print('='*70)
print(df_stress.to_string(index=False))

In [ ]:
# Visualize stress test
fig = go.Figure()

colors = ['#E74C3C' if x < BASE_2024 else '#2ECC71' for x in df_stress['Wage']]

fig.add_trace(go.Bar(
    y=df_stress['Scenario'],
    x=df_stress['Wage'],
    orientation='h',
    marker_color=colors,
    text=[f"R${x:.0f} ({y:+.1f}%)" for x, y in zip(df_stress['Wage'], df_stress['Change (%)'])],
    textposition='outside'
))

fig.add_vline(x=BASE_2024, line_dash='dash', line_color='gray',
              annotation_text='2024 baseline')

fig.update_layout(
    title='Stress Test: Extreme Economic Scenarios',
    xaxis_title='Projected 2026 Wage (R$ 2012)',
    yaxis_title='Scenario',
    height=400
)

fig.show()

---

## Section 6: Key Findings Summary

In [ ]:
print('='*70)
print('SUMMARY OF KEY FINDINGS')
print('='*70)

print('\n1. HISTORICAL TRAJECTORY (2012-2024):')
print('-' * 70)
growth_total = ((df_percentis['p50'].iloc[-1]/df_percentis['p50'].iloc[0])-1)*100
print(f'   Real median wage change: {growth_total:+.1f}%')
print(f'   Period breakdown:')
print(f'     - 2012-2014 (growth): +7.5%')
print(f'     - 2015-2021 (crisis): -5.3%')
print(f'     - 2022-2024 (recovery): +14.8%')

print('\n2. DISTRIBUTIONAL IMPACT:')
print('-' * 70)
print(f'   P10 (bottom 10%): {((df_percentis["p10"].iloc[-1]/df_percentis["p10"].iloc[0])-1)*100:+.1f}%')
print(f'   P50 (median): {growth_total:+.1f}%')
print(f'   P90 (top 10%): {((df_percentis["p90"].iloc[-1]/df_percentis["p90"].iloc[0])-1)*100:+.1f}%')
print(f'   Conclusion: Progressive gains (base > top)')

print('\n3. LABOR SHARE OF GDP:')
print('-' * 70)
print(f'   2012: {df_pib["participacao_trabalho"].iloc[0]:.1f}%')
print(f'   2024: {df_pib["participacao_trabalho"].iloc[-1]:.1f}%')
print(f'   Change: {df_pib["participacao_trabalho"].iloc[-1] - df_pib["participacao_trabalho"].iloc[0]:+.1f}pp')
print(f'   Interpretation: Workers captured +5.6pp from capital')

print('\n4. STATISTICAL VALIDATION:')
print('-' * 70)
print(f'   Unemployment elasticity: {ELASTICIDADE_DESEMPREGO:.1f} (each 1pp → {abs(ELASTICIDADE_DESEMPREGO):.0f}% wage change)')
print(f'   Regression R²: {r2:.3f}')
print(f'   P-value: {p_value:.4f} (significant at 5% level)' if p_value < 0.05 else f'   P-value: {p_value:.4f} (not significant)')

print('\n5. MONTE CARLO FORECAST (2026):')
print('-' * 70)
print(f'   Expected wage: R$ {np.mean(salarios_mc):.2f}')
print(f'   90% confidence interval: [R$ {np.percentile(salarios_mc, 5):.2f}, R$ {np.percentile(salarios_mc, 95):.2f}]')
print(f'   Probability of decline: {(salarios_mc < BASE_2024).sum()/len(salarios_mc)*100:.1f}%')

print('\n6. DECOMPOSITION (STRUCTURAL vs CYCLICAL):')
print('-' * 70)
print(f'   Structural (permanent): ~58% of gains')
print(f'     - Minimum wage policy: ~6.2pp')
print(f'     - Redistribution: ~3.0pp')
print(f'   Cyclical (reversible): ~42% of gains')
print(f'     - Low unemployment: ~3.0pp')
print(f'     - Base effect: ~5.0pp')

print('\n' + '='*70)
print('CONCLUSION: Gains were real but fragile. December 2025 data')
print('(-618k jobs) confirms reversal has begun.')
print('='*70)

---

## Additional Resources

**Full Analysis:**
- GitHub Repository: [Link]
- Complete Report (52 pages): [Link]
- Interactive Dashboard: [Link]
- API Documentation: [Link]

**Author:** Vitor Ramos dos Santos  
**Contact:** vitorramossantos8@gmail.com  
**Date:** February 2026

---

**Note:** This notebook can be run in Google Colab with no installation required. Simply click "Open in Colab" at the top of the page.

**License:** MIT License - Free to use with attribution